In [1]:

from __future__ import annotations

import geopandas as gpd
from libpysal.weights import Queen

from dissmodel.geo import SpatialModel
from dissmodel.visualization import track_plot


# ── tabela_usos ───────────────────────────────────────────────────────────────
MANGUE                    = 1
VEGETACAO_TERRESTRE       = 2
MAR                       = 3
AREA_ANTROPIZADA          = 4
SOLO_DESCOBERTO           = 5
SOLO_INUNDADO             = 6
AREA_ANTROPIZADA_INUNDADA = 7
MANGUE_MIGRADO            = 8
MANGUE_INUNDADO           = 9
VEG_TERRESTRE_INUNDADA    = 10

USOS_INUNDADOS: list[int] = [
    MAR, SOLO_INUNDADO, AREA_ANTROPIZADA_INUNDADA,
    MANGUE_INUNDADO, VEG_TERRESTRE_INUNDADA,
]

# seco → inundado (Bezerra 2014)
REGRAS_INUNDACAO: dict[int, int] = {
    MANGUE:              MANGUE_INUNDADO,
    MANGUE_MIGRADO:      MANGUE_INUNDADO,
    VEGETACAO_TERRESTRE: VEG_TERRESTRE_INUNDADA,
    AREA_ANTROPIZADA:    AREA_ANTROPIZADA_INUNDADA,
    SOLO_DESCOBERTO:     SOLO_INUNDADO,
}

USO_LABELS: dict[int, str] = {
    MANGUE:                    "Mangue",
    VEGETACAO_TERRESTRE:       "Vegetação Terrestre",
    MAR:                       "Mar",
    AREA_ANTROPIZADA:          "Área Antropizada",
    SOLO_DESCOBERTO:           "Solo Descoberto",
    SOLO_INUNDADO:             "Solo Inundado",
    AREA_ANTROPIZADA_INUNDADA: "Área Antrop. Inundada",
    MANGUE_MIGRADO:            "Mangue Migrado",
    MANGUE_INUNDADO:           "Mangue Inundado",
    VEG_TERRESTRE_INUNDADA:    "Veg. Terrestre Inundada",
}

# cores exatas do Lua (tabela_usos RGB → hex)
USO_COLORS: dict[int, str] = {
    MANGUE:                    "#006400",
    VEGETACAO_TERRESTRE:       "#808000",
    MAR:                       "#00008b",
    AREA_ANTROPIZADA:          "#ffd700",
    SOLO_DESCOBERTO:           "#ffdead",
    SOLO_INUNDADO:             "#000000",
    AREA_ANTROPIZADA_INUNDADA: "#323232",
    MANGUE_MIGRADO:            "#00ff00",
    MANGUE_INUNDADO:           "#ff0000",
    VEG_TERRESTRE_INUNDADA:    "#000000",
}

# ── tabela_solos ──────────────────────────────────────────────────────────────
SOLO_CANAL_FLUVIAL  = 0
SOLO_MANGUE         = 3
SOLO_MANGUE_MIGRADO = 9
SOLO_OUTROS         = 4

SOLO_LABELS: dict[int, str] = {
    SOLO_CANAL_FLUVIAL:  "Canal Fluvial",
    SOLO_MANGUE:         "Mangue",
    SOLO_MANGUE_MIGRADO: "Mangue Migrado",
    SOLO_OUTROS:         "Outros",
}

# ── geografia — Ilha do Maranhão ──────────────────────────────────────────────
ORIGIN_X  = 500_000.0    # UTM Easting  (SIRGAS 2000 / UTM 24S)
ORIGIN_Y  = 9_700_000.0  # UTM Northing
CRS       = "EPSG:31984"
CELL_SIZE = 100.0         # metros

# ── GeoTIFF: especificação de bandas (nome, dtype numpy, nodata) ──────────────
TIFF_BANDS: list[tuple[str, str, float]] = [
    ("uso",  "int16",   0),
    ("alt",  "float32", -9999.0),
    ("solo", "int16",   -1),
]

# cores da tabela_solos (para RasterMap)
SOLO_COLORS: dict[int, str] = {
    SOLO_CANAL_FLUVIAL:  "#0000ff",   # azul — canal de drenagem
    SOLO_MANGUE:         "#006400",   # verde escuro
    SOLO_MANGUE_MIGRADO: "#228b22",   # verde floresta
    SOLO_OUTROS:         "#888888",   # cinza
}

@track_plot("flooded_cells", "blue")
class FloodModel(SpatialModel):
    """
    Hydrological model implemented with DisSModel + GeoDataFrame.

    Equivalence with the raster version
    -----------------------------------
    RasterBackend.shift2d()          →  neighs_id(idx) / neighbor_values()
    np.isin(uso, USOS_INUNDADOS)     →  uso_past.isin(USOS_INUNDADOS)
    loop over DIRS_MOORE             →  loop over real GDF neighbors
    vectorized over full grid        →  cell-by-cell loop (slower,
                                        but faithful to real geometry)

    Parameters
    ----------
    gdf           : GeoDataFrame with columns attr_uso and attr_alt
    taxa_elevacao : meters/year — IPCC RCP8.5 ≈ 0.011
    attr_uso      : land-use column. Default: "uso"
    attr_alt      : elevation column. Default: "alt"
    """


    

    def setup(
        self,
        taxa_elevacao: float = 0.011,
        attr_uso:      str   = "uso",
        attr_alt:      str   = "alt",
    ) -> None:
        self.taxa_elevacao = taxa_elevacao
        self.attr_uso      = attr_uso
        self.attr_alt      = attr_alt

        # metrics exposed for @track_plot / Chart
        self.flooded_cells = 0
        self.novas_inundadas   = 0
        self.nivel_mar_atual   = 0.0

        # Queen = Moore neighborhood (8 directions) for regular grids
        # silence_warnings suppresses island warnings (cells without neighbors)
        self.create_neighborhood(strategy=Queen, silence_warnings=True)

    def execute(self) -> None:
        nivel_mar = self.env.now() * self.taxa_elevacao

        # Snapshots — equivalent to cell.past[] in TerraME
        uso_past = self.gdf[self.attr_uso].copy()
        alt_past = self.gdf[self.attr_alt].copy()

        # ── sources: isSeaOrFlooded(uso) and alt >= 0 ─────────────────────────
        fontes = set(
            uso_past.index[
                uso_past.isin(USOS_INUNDADOS) & (alt_past >= 0)
            ]
        )

        # ── A. Elevation — flow diffusion (relative condition) ────────────────
        # Lua: if neighbor.past[alt] <= currentAlt: neigh[alt] += flow
        alt_nova = alt_past.copy()

        for idx in fontes:
            alt_atual = alt_past[idx]
            vizinhos  = self.neighs_id(idx)

            viz_baixos = 1 + sum(
                1 for n in vizinhos if alt_past[n] <= alt_atual
            )
            fluxo = self.taxa_elevacao / viz_baixos

            alt_nova[idx] += fluxo
            for n in vizinhos:
                if alt_past[n] <= alt_atual:
                    alt_nova[n] += fluxo

        self.gdf[self.attr_alt] = alt_nova

        # ── B. Flooding — absolute elevation threshold ───────────────────────
        # Lua: if neighbor.past[alt] <= seaLevel and not isSeaOrFlooded(neigh):
        #          applyFlooding(neighbor)
        # Uses alt_past — faithful to TerraME .past semantics
        uso_novo = uso_past.copy()

        for idx in self.gdf.index:
            uso_atual = uso_past[idx]
            if uso_atual not in REGRAS_INUNDACAO:
                continue
            if alt_past[idx] > nivel_mar:
                continue
            if any(n in fontes for n in self.neighs_id(idx)):
                uso_novo[idx] = REGRAS_INUNDACAO[uso_atual]

        self.gdf[self.attr_uso] = uso_novo

        # ── metrics ──────────────────────────────────────────────────────────
        inund = uso_novo.isin(USOS_INUNDADOS) & (uso_novo != MAR)
        novas = uso_novo.isin(USOS_INUNDADOS) & ~uso_past.isin(USOS_INUNDADOS)

        self.flooded_cells = int(inund.sum())
        self.novas_inundadas   = int(novas.sum())
        self.nivel_mar_atual   = round(nivel_mar, 4)

In [2]:
FloodModel

__main__.FloodModel

In [4]:
from minio import Minio
import io, geopandas as gpd

from getpass import getpass
secret  = "inpe_secret_2024" # = getpass("Secret key: ")

minio = Minio("minio:9000",
              access_key="inpe_admin",
              secret_key=secret,
              secure=False)

# baixa o zip do MinIO para memória
obj  = minio.get_object("dissmodel-inputs", "synthetic_grid_60x60_shp.zip")
data = io.BytesIO(obj.read())

# GeoPandas lê zip com shapefile diretamente
gdf = gpd.read_file(data)
gdf.head()

,row,col,uso,uso_nome,alt,passo,nivel_mar,solo,solo_nome,geometry
0,0,0,3,Mar,0.0,0,0.0,4,Outros,"POLYGON ((500100 9705900, 500000 9705900, 5000..."
1,0,1,3,Mar,0.0,0,0.0,4,Outros,"POLYGON ((500200 9705900, 500100 9705900, 5001..."
2,0,2,3,Mar,0.0,0,0.0,4,Outros,"POLYGON ((500300 9705900, 500200 9705900, 5002..."
3,0,3,3,Mar,0.0,0,0.0,4,Outros,"POLYGON ((500400 9705900, 500300 9705900, 5003..."
4,0,4,3,Mar,0.0,0,0.0,4,Outros,"POLYGON ((500500 9705900, 500400 9705900, 5004..."


In [5]:
import pathlib
import geopandas as gpd

from matplotlib.colors import ListedColormap, BoundaryNorm

from dissmodel.core import Environment
from dissmodel.visualization import Map, Chart

# ── parâmetros ─────────────────────────────────────────────
shp_path = "flood_model.shp"   # ajuste aqui
taxa_elevacao = 0.5
altura_mare = 6.0
END_TIME = 10

attr_uso  = "uso"
attr_alt  = "alt"
attr_solo = "solo"

show_chart = False
save = True


# ── colormaps (se já definidos, pode remover) ──────────────
_vals = sorted(USO_COLORS)
USO_CMAP = ListedColormap([USO_COLORS[k] for k in _vals])
USO_NORM = BoundaryNorm([v - 0.5 for v in _vals] + [_vals[-1] + 0.5], USO_CMAP.N)

# ── ambiente ───────────────────────────────────────────────
env = Environment(start_time=1, end_time=END_TIME)

# ── modelo ─────────────────────────────────────────────────
FloodModel(
    gdf=gdf,
    taxa_elevacao=taxa_elevacao,
    attr_uso=attr_uso,
    attr_alt=attr_alt,
)

# ── visualização ───────────────────────────────────────────
Map(
    gdf=gdf,
    plot_params={
        "column": attr_uso,
        "cmap": USO_CMAP,
        "norm": USO_NORM,
        "legend": False
    }
)

if show_chart:
    Chart(select={"flooded_cells", "mangrove_migrated"})

# ── execução ───────────────────────────────────────────────
env.run()

bucket_out = "dissmodel-outputs"

# ── salvar (GeoDataFrame → memória → MinIO) ────────────────
buffer = io.BytesIO()

# ── salvar ─────────────────────────────────────────────────
# salva como GPKG em memória
gdf.to_file(buffer, driver="GPKG", layer="result")
buffer.seek(0)

# envia para MinIO
minio.put_object(
    bucket_out,
    "simulation_result.gpkg",
    data=buffer,
    length=buffer.getbuffer().nbytes,
    content_type="application/geopackage+sqlite3"
)

print("✅ Resultado salvo no MinIO")

Output()

Running from 1 to 10 (duration: 9)
✅ Resultado salvo no MinIO
